# Machine Learning Final Proyect - Nopales
### David Villanueva Guzmán - U6110618
### Alvaro Cámara Guerra - U6110583

## CROSS VALIDATION 

### Import Required Libraries

In [ ]:
import sys
print(sys.executable)
import torch
print(torch.__version__)
print(torch.cuda.is_available())    

In [ ]:
import torch, kornia, platform
# Define device foor runtime
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Torch:", torch.__version__)
print("CUDA version in torch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Kornia:", kornia.__version__)
print("Python:", platform.python_version())
print(device)

In [ ]:
!nvidia-smi

In [ ]:
# !pip install "numpy<2.0" --force-reinstall
# !pip install kornia
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# For data processing
# !pip install pandas
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Load the Dataset
Milan Dataset --

In [ ]:
# Import Multiclass and Binary annotations
file_path_MC = r"C:\Users\kartp\OneDrive\Documentos\IFRoS\1st semester\Machine Learning\Labs\FP\annotations_multiclass.csv"
file_path_BIN = r"C:\Users\kartp\OneDrive\Documentos\IFRoS\1st semester\Machine Learning\Labs\FP\annotations_binary.csv"
# file_path_MC = "/home/udem09/ifros/machine-learning/annotations_multiclass.csv"
# file_path_BIN = "/home/udem09/ifros/machine-learning/annotations_binary.csv"

# Get dataframes of annotations
df_MC = pd.read_csv(file_path_MC)
df_MC.drop(columns=["Unnamed: 2", "Unnamed: 3"], inplace=True)

print(df_MC.head(10))

#### Data Preprocessing

In [ ]:
mapping = {
    "0_0": 0,
    "0_1": 1,
    "1_0": 2,
    "1_1": 3
}

df_MC["printError_physicalDefect_int"] = df_MC["printError_physicalDefect"].map(mapping)

# Count occurrences of each class
print_counts = df_MC["printError_physicalDefect_int"].value_counts().sort_index()

# Create the figure
fig, ax = plt.subplots(figsize=(7, 5))

# Define x positions for side-by-side bars
x = range(len(print_counts))  # same class positions (e.g. 0 and 1)
width = 0.5  # bar width

# Plot bars
bars1 = ax.bar([i - width/2 for i in x], print_counts, width, color='#1f77b4', alpha=0.8)

# Add text labels on top of bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 10, f'{int(height)}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Customize axes
ax.set_xlabel("Class")
ax.set_ylabel("Counts")
ax.set_title("Distribution of Print and Physical Errors")
ax.set_xticks(x)
ax.set_xticklabels(print_counts.index)  # e.g. 0 and 1 only
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(df_MC)

### Binary annotations

In [ ]:
# Lista de strings a buscar en el nombre de la imagen

# Filtrar solo filas donde la segunda columna sea exactamente '0_0'
df_MC["binary"] = (df_MC["printError_physicalDefect_int"] != 0).astype(int)

# Count occurrences of each class
print_counts = df_MC["binary"].value_counts().sort_index()

# Create the figure
fig, ax = plt.subplots(figsize=(7, 5))

# Define x positions for side-by-side bars
x = range(len(print_counts))  # same class positions (e.g. 0 and 1)
width = 0.5  # bar width

# Plot bars
bars1 = ax.bar([i - width/2 for i in x], print_counts, width, label='Print Error', color='#1f77b4', alpha=0.8)

# Add text labels on top of bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 10, f'{int(height)}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Customize axes
ax.set_xlabel("Class")
ax.set_ylabel("Counts")
ax.set_title("Distribution of Print and Physical Errors")
ax.set_xticks(x)
ax.set_xticklabels(print_counts.index)  # e.g. 0 and 1 only
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Data reduction

#### Unzip images inside google drive

In [ ]:
# extract_path = "/home/udem09/ifros/machine-learning/arrays"
# extract_path = '/content/drive/MyDrive/IFRoS/ML Labs/MILAN-dataset/arrays'
extract_path = r"C:\Users\kartp\OneDrive\Documentos\IFRoS\1st semester\Machine Learning\Labs\FP\drive-download-20251113T114126Z-1-001\arrays"

#### Define NPYImageDataset class to load the images

In [ ]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np

class NPYImageDataset(Dataset):
    def __init__(self, root_dir, names, labels, transform=None, selected_channels=None):
        """
        root_dir : Folder with .npy files
        names    : List like ['003_045_012', '002_047_12', ...]
        labels   : Array-like with class labels aligned with 'names'
        transform: torchvision transform
        selected_channels: List of channel indices to keep (e.g., [0, 3, 5])
                          If None, uses all 6 channels
        """
        self.root_dir = root_dir
        self.names = names
        self.labels = labels
        self.transform = transform
        self.selected_channels = selected_channels

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        filename = f"{name}.npy"
        path = os.path.join(self.root_dir, filename)

        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")

        img = np.load(path)                          # (H, W, C)
        img = torch.from_numpy(img).permute(2, 0, 1).float()  # (C, H, W)

        # Select specific channels if specified
        if self.selected_channels is not None:
            img = img[self.selected_channels, :, :]

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(self.labels[idx]).long()

        return img, label
    
# GRAY, AM, G, R, W and NORMAL
# model_channels = [0, 2, 3, 4] # GRAY, G, R, W
model_channels = [0, 2, 3] # AM, G, R, W
# model_channels = [0, 1, 2, 3, 4, 5] # All the channels
num_channels = len(model_channels)

### Define all the labels and all the images names

In [ ]:
all_names = df_MC["file_prefix"].values      # list of .npy filenames
all_labels_list = df_MC["binary"].values       # 0 / 1 labels

## Save some images to test the model and ensure consistency with pytorch + ONNX

In [ ]:
from sklearn.model_selection import train_test_split

# All indices as ints
indices = np.arange(len(all_names))

# dataset + pseudo-test
dataset_idx, test_idx = train_test_split(
    indices,
    test_size=0.05,
    stratify=all_labels_list,
    random_state=42
)

In [ ]:
# Apply split to names and labels
dataset_names = all_names[dataset_idx]
test_names   = all_names[test_idx]

dataset_labels = all_labels_list[dataset_idx]
test_labels   = all_labels_list[test_idx]

In [ ]:
print(len(dataset_names))
print(len(all_labels_list))

#### Data transforming

In [ ]:
import torchvision.transforms.v2 as transforms
import torch
import kornia.augmentation as K

def scale_image(x):
    return x / 255.0

data_transform = transforms.Compose([
    # Input comes as (C,H,W) torch tensor from NPY
    # Scale to [0,1]
    transforms.Lambda(scale_image),
    transforms.ToDtype(torch.float32),

    # Normalize all 6 channels
    transforms.Normalize(
        mean=[0.5]*num_channels,
        std=[0.5]*num_channels
    )
])

test_transform = transforms.Compose([
    transforms.Lambda(scale_image),
    transforms.ToDtype(torch.float32),

    transforms.Normalize(
        mean=[0.5]*6,
        std=[0.5]*6
    )
])

## Data augmentation

In [ ]:
kornia_augment = torch.nn.Sequential(
    # === Geometric Augmentations (Critical for defects) ===
    # Defects can appear in any orientation
    K.RandomHorizontalFlip(p=0.5),
    K.RandomVerticalFlip(p=0.5),
    K.RandomRotation(degrees=180, p=0.6),  # Full rotation since defects are orientation-agnostic
    
    # Small affine transforms (simulate camera position variation)
    K.RandomAffine(
        degrees=0,
        translate=(0.08, 0.08),  # Slight translations
        scale=(0.95, 1.05),      # Minor zoom
        shear=5,                 # Small shear
        p=0.4
    ),
    
    # === Photometric Augmentations (Simulate lighting/camera variations) ===
    # Lighting changes are CRITICAL for your multi-channel setup
    K.RandomBrightness(brightness=(0.7, 1.3), p=0.6),
    K.RandomContrast(contrast=(0.7, 1.3), p=0.6),
    # K.RandomGamma(gamma=(0.8, 1.2), p=0.5),
    
    # Simulate different camera exposures
    # K.ColorJitter(
    #     brightness=0.2,
    #     contrast=0.2,
    #     saturation=0.1,  # Minimal since mostly grayscale
    #     hue=0.0,         # No hue shift for grayscale
    #     p=0.4
    # ),
    
    # === Noise Augmentations (Sensor/acquisition noise) ===
    K.RandomGaussianNoise(mean=0., std=0.02, p=0.3),
    
    # Simulate slight camera blur/defocus
    K.RandomGaussianBlur(
        kernel_size=(3, 3),
        sigma=(0.1, 1.5),
        p=0.25
    ),
    
    # === Spatial Augmentations (Test robustness) ===
    # Elastic deformation (minimal, simulates slight surface warping)
    K.RandomElasticTransform(
        kernel_size=(33, 33),
        sigma=(4.0, 4.0),
        alpha=(0.5, 0.5),
        p=0.2
    ),
    
    # Random erasing (helps with occlusion robustness)
    # Small patches since defects are localized
    K.RandomErasing(
        scale=(0.01, 0.04),  # Very small erasures
        ratio=(0.3, 3.3),
        value=0.5,  # Gray value
        p=0.15
    ),
)

In [ ]:
dataset = NPYImageDataset(
    root_dir=extract_path,
    names=dataset_names,
    labels=dataset_labels,
    transform=None,
    selected_channels=model_channels
)

test_dataset = NPYImageDataset(
    root_dir=extract_path,
    names=test_names,
    labels=test_names,
    transform=None,
    selected_channels=model_channels
)

#### Visuazlie a sample and the whole dataset

In [ ]:
import os
import numpy as np

bad_files = []
good = 0

for f in os.listdir(extract_path):
    if not f.lower().endswith(".npy"):
        continue

    full = os.path.join(extract_path, f)

    try:
        arr = np.load(full)
        _ = arr.shape  # force read
        good += 1
    except Exception as e:
        print("❌ Bad file:", full)
        print("   Error:", repr(e))
        bad_files.append(full)

print("\n✅ Good files:", good)
print("❌ Bad files:", len(bad_files))


In [ ]:
# Visualize one sample directly from the DATASET 
import matplotlib.pyplot as plt

# Get one raw sample from the dataset
visualize_dataset = NPYImageDataset(
    root_dir=extract_path,
    names=dataset_names,
    labels=dataset_labels,
    transform=data_transform,
    selected_channels=model_channels
)

img, label = dataset[0]
print("label = ",label)
img = kornia_augment(img)

print("Type:", type(img))
print("Shape:", img.shape)      # (1, 6, H, W)
print("Label:", label)

# Undo normalization for visualization
upper_index = 3 if img.shape[1] >= 6 else img.shape[1]
img_vis = img[0, :upper_index] * 0.5 + 0.5   # Take first 3 channels from the first image in batch as pseudo-RGB
img_vis = img_vis.clamp(0, 1)  

# Show image
plt.imshow(img_vis.permute(1, 2, 0))
plt.title("Train Sample (Pseudo-RGB)")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def get_channels_names(indices):
    """
    Return channel labels given a list of channel indices.

    Channel order:
    0: GRAY
    1: AM
    2: G
    3: R
    4: W
    5: NORMAL
    """
    channel_labels = ["GRAY", "AM", "G", "R", "W", "NORMAL"]
    return [channel_labels[i] for i in indices]

def analyze_channel_correlation(dataset):
    """Check if channels are redundant"""
    # Sample 300 random images
    sample_images = []
    for i in np.random.choice(len(dataset), 300, replace=False):
        img, _ = dataset[i]  # [6, H, W]
        sample_images.append(img.numpy())
    
    sample_images = np.array(sample_images)  # [300, 6, H, W]
    
    # Flatten spatial dimensions
    samples_flat = sample_images.reshape(300, num_channels, -1).mean(axis=2)  # [300, 6]
    
    # Correlation matrix
    corr_matrix = np.corrcoef(samples_flat.T)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', 
                xticklabels=get_channels_names(model_channels),
                yticklabels=get_channels_names(model_channels),
                cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title('Channel Correlation Matrix')
    plt.tight_layout()
    plt.savefig('channel_correlation.png', dpi=150)
    plt.show()
    
    print("\nCorrelation Matrix:")
    print(corr_matrix)
    
    # Check for highly correlated pairs (> 0.85)
    high_corr = np.where((corr_matrix > 0.85) & (corr_matrix < 1.0))
    if len(high_corr[0]) > 0:
        print("\n⚠️ Highly correlated channel pairs (>0.85):")
        for i, j in zip(high_corr[0], high_corr[1]):
            if i < j:
                channels = get_channels_names(model_channels)
                print(f"  {channels[i]} <-> {channels[j]}: {corr_matrix[i,j]:.3f}")
    
    return corr_matrix

# Run it
corr = analyze_channel_correlation(dataset)

### Setting the Device

### Train and Validation Fucntions

In [ ]:
#train function
from sklearn.metrics import average_precision_score, f1_score
import torch.nn.functional as F
import numpy as np

def Train(model, train_loader, optimizer, criterion):
    model.train()

    running_loss = 0.0
    all_probs = []
    all_labels = []

    for i, (images, labels) in enumerate(train_loader):


        # Move augmented batch to GPU for training
        images = images.to(device)
        labels = labels.to(device)

        # Data augmentation with kornia in the GPU 
        images = kornia_augment(images)

        # Initialize backpropagation and optimization
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Backpropagation
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Convert logits → probability of positive class
        probs = F.softmax(outputs, dim=1)[:, 1]

        all_probs.extend(probs.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    # Compute AUC-PR for the full epoch
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # AUC-PR
    if len(np.unique(all_labels)) > 1:
        auc_pr = average_precision_score(all_labels, all_probs)
    else:
        auc_pr = 0.0

    # Average loss
    avg_loss = running_loss / len(train_loader)

    # Binary predictions for F1
    preds = (all_probs >= 0.5).astype(int)

    # F1-score
    f1 = f1_score(all_labels, preds)

    return avg_loss, auc_pr, f1

#validation function
def Validate(model, val_loader, criterion):
    model.eval()

    running_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            # Convert logits → probability of positive class
            probs = F.softmax(outputs, dim=1)[:, 1]

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

        # AUC-PR
    if len(np.unique(all_labels)) > 1:
        auc_pr = average_precision_score(all_labels, all_probs)
    else:
        auc_pr = 0.0

    avg_loss = running_loss / len(val_loader)

    # ----- FIND BEST THRESHOLD FOR F1 -----
    thresholds = np.linspace(0.0, 1.0, 200)
    f1_scores = [
        f1_score(all_labels, (all_probs >= t).astype(int)) 
        for t in thresholds
    ]
    best_idx = int(np.argmax(f1_scores))
    best_threshold = float(thresholds[best_idx])
    best_f1 = float(f1_scores[best_idx])

    return avg_loss, auc_pr, best_f1, best_threshold

### Import the model 

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models


def build_model_resnet34(
    head,
    dropout_rate=0.35,
    frozen_layers=(""), # No initial frozen layer
    pretrained=True
  ):
    """
    Creates a fresh ResNet34 with:
    - ImageNet pretrained weights
    - 6-channel input
    - Custom MLP head
    - Selective layer freezing

    Safe to call once per CV fold.
    """

    # Load the model RESNET34
    model = models.resnet34(
        weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None,
        progress=False
    )

    # Adapts the first layer
    old_conv = model.conv1
    new_conv = nn.Conv2d(
        in_channels=6,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=old_conv.bias
    )

    with torch.no_grad():
        # Copy RGB weights
        new_conv.weight[:, :3] = old_conv.weight
        # Initialize extra channels sensibly
        new_conv.weight[:, 3:] = old_conv.weight.clone()

    model.conv1 = new_conv

    # Freeze selected layers
    for name, param in model.named_parameters():
        param.requires_grad = not name.startswith(frozen_layers)

    # Adapt the FC layer
    model.fc = head
    # Return the virgin model 
    return model

In [ ]:
def build_model_resnet18(
    head,
    dropout_rate=0.35,
    frozen_layers = (""), # No initial frozen layer by default
    pretrained: bool =True,
    n_channels: int = 6,
    selected_channels: list[int] = None
  ):
    """
    Creates a fresh ResNet34 with:
    - ImageNet pretrained weights
    - 6-channel input
    - Custom MLP head
    - Selective layer freezing

    Safe to call once per CV fold.
    """

    def get_channel_name(idx):
        """Helper to get channel name from index"""
        names = ['GRAY', 'AM', 'G', 'R', 'W', 'NORMAL']
        return names[idx] if 0 <= idx < len(names) else f'CH{idx}'
    
    # Load the model RESNET18
    model = models.resnet18(
        weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    )

    # Adapts the first layer
    old_conv = model.conv1
    new_conv = nn.Conv2d(
        in_channels=n_channels,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=old_conv.bias
    )

    # Only redefine the first layer if the number of input channels is different than 3 (original input size)
    with torch.no_grad():
        if n_channels == 3 and selected_channels is None:
            # Direct RGB copy (standard case)
            new_conv.weight = old_conv.weight
            
        elif selected_channels is not None and pretrained:
            # Smart mapping: map pretrained RGB weights to corresponding channels
            
            # Define channel-to-RGB mapping
            channel_mapping = {
                0: 0,  # GRAY → R channel (grayscale approximates red)
                1: 0,  # AM → R channel
                2: 1,  # G → G channel (GREEN MATCH)
                3: 0,  # R → R channel (RED MATCH)
                4: 0,  # W → R channel (white/grayscale)
                5: 1,  # NORMAL → G channel
            }
            
            # Initialize with random small weights (fallback)
            nn.init.kaiming_normal_(new_conv.weight, mode='fan_out', nonlinearity='relu')
            
            # Map pretrained weights to corresponding channels
            for new_idx, orig_channel_idx in enumerate(selected_channels):
                if orig_channel_idx in channel_mapping:
                    # Get which RGB channel to use
                    rgb_idx = channel_mapping[orig_channel_idx]
                    
                    # Copy pretrained weights from that RGB channel
                    new_conv.weight[:, new_idx:new_idx+1, :, :] = old_conv.weight[:, rgb_idx:rgb_idx+1, :, :]
                    
                    print(f"  Channel {new_idx}: {get_channel_name(orig_channel_idx)} ← RGB[{rgb_idx}] ({'R' if rgb_idx==0 else 'G' if rgb_idx==1 else 'B'})")
                else:
                    print(f"  Channel {new_idx}: {get_channel_name(orig_channel_idx)} ← Random init")
        
        else:
            # Fallback: use RGB mean for all channels
            rgb_mean = old_conv.weight.mean(dim=1, keepdim=True)
            new_conv.weight[:, :] = rgb_mean.repeat(1, n_channels, 1, 1)
            print(f"  All {n_channels} channels initialized with RGB mean")
    
    model.conv1 = new_conv

    # Freeze selected layers
    for name, param in model.named_parameters():
        param.requires_grad = not name.startswith(frozen_layers)

    # Adapt the FC layer
    model.fc = head

    # Return the virgin model 
    return model

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_model_efficientnet_b0(
    head,
    dropout_rate=0.4,  # EfficientNet benefits from higher dropout
    frozen_layers=(""),
    pretrained: bool = True,
    n_channels: int = 6,
    selected_channels: list[int] = None
):
    """
    Creates EfficientNet-B0 with:
    - ImageNet pretrained weights
    - Multi-channel input (6 channels)
    - Custom classification head
    - Selective layer freezing
    
    Args:
        head: Custom classifier head (nn.Module)
        dropout_rate: Dropout rate for regularization
        frozen_layers: Tuple of layer prefixes to freeze initially
        pretrained: Whether to use ImageNet pretrained weights
        n_channels: Number of input channels (default: 6)
        selected_channels: List of channel indices to use (None = use all)
    
    Returns:
        Modified EfficientNet-B0 model
    """
    
    def get_channel_name(idx):
        """Helper to get channel name from index"""
        names = ['GRAY', 'AM', 'G', 'R', 'W', 'NORMAL']
        return names[idx] if 0 <= idx < len(names) else f'CH{idx}'
    
    # Load EfficientNet-B0
    model = efficientnet_b0(
        weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    )
    
    # Adapt the first convolutional layer
    # EfficientNet structure: features[0][0] is the first Conv2d
    old_conv = model.features[0][0]
    
    new_conv = nn.Conv2d(
        in_channels=n_channels,
        out_channels=old_conv.out_channels,  # 32 for EfficientNet-B0
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False  # EfficientNet doesn't use bias in first conv
    )
    
    # Initialize weights intelligently
    with torch.no_grad():
        if n_channels == 3 and selected_channels is None:
            # Direct RGB copy (standard case)
            new_conv.weight.copy_(old_conv.weight)
            print("  Using standard RGB weights")
            
        elif selected_channels is not None and pretrained:
            # Smart mapping: map pretrained RGB weights to corresponding channels
            
            # Channel-to-RGB mapping for your eraser inspection system
            channel_mapping = {
                0: 0,  # GRAY → R channel (grayscale → red)
                1: 0,  # AM → R channel
                2: 1,  # G → G channel (GREEN MATCH)
                3: 0,  # R → R channel (RED MATCH)
                4: 0,  # W → R channel (white/grayscale)
                5: 1,  # NORMAL → G channel
            }
            
            # Initialize with Kaiming normal (better for ReLU activations)
            nn.init.kaiming_normal_(new_conv.weight, mode='fan_out', nonlinearity='relu')
            
            print(f"  Initializing {len(selected_channels)}-channel input:")
            
            # Map pretrained weights to selected channels
            for new_idx, orig_channel_idx in enumerate(selected_channels):
                if orig_channel_idx in channel_mapping:
                    rgb_idx = channel_mapping[orig_channel_idx]
                    
                    # Copy pretrained weights from corresponding RGB channel
                    new_conv.weight[:, new_idx:new_idx+1, :, :].copy_(
                        old_conv.weight[:, rgb_idx:rgb_idx+1, :, :]
                    )
                    
                    rgb_name = 'R' if rgb_idx == 0 else 'G' if rgb_idx == 1 else 'B'
                    print(f"    Channel {new_idx}: {get_channel_name(orig_channel_idx)} ← RGB[{rgb_idx}] ({rgb_name})")
                else:
                    print(f"    Channel {new_idx}: {get_channel_name(orig_channel_idx)} ← Random init")
        
        else:
            # Fallback: use RGB mean across all channels
            rgb_mean = old_conv.weight.mean(dim=1, keepdim=True)
            new_conv.weight.copy_(rgb_mean.repeat(1, n_channels, 1, 1))
            print(f"  All {n_channels} channels initialized with RGB mean")
    
    # Replace first convolution
    model.features[0][0] = new_conv
    
    # Freeze selected layers
    for name, param in model.named_parameters():
        should_freeze = any(name.startswith(frozen) for frozen in frozen_layers)
        param.requires_grad = not should_freeze
    
    # Replace classifier head
    # EfficientNet-B0 has a 1280-dimensional feature before classifier
    model.classifier = head
    
    return model

## Criterion and optimizer

In [ ]:
# FocalLoss alternative as criterion
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss para clasificación binaria/multiclase con desbalance.

    Parámetros:
    -----------
    alpha : tensor o None
        Peso para cada clase. Ej: [0.3, 0.7] da más peso a clase 1
    gamma : float
        Factor de enfoque. Típicamente 2.0
        - gamma=0 equivale a CrossEntropyLoss
        - gamma>0 reduce la pérdida para ejemplos bien clasificados
    label_smoothing : float
        Suavizado de etiquetas (0.0 a 0.2 típicamente)
    """
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # será un tensor en GPU
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        inputs: logits del modelo, shape (N, C) donde C=num_classes
        targets: etiquetas, shape (N,) con valores 0 o 1
        """
        # Calcular log-softmax para estabilidad numérica
        log_probs = F.log_softmax(inputs, dim=1)

        # Obtener log_prob de la clase correcta para cada muestra
        # targets.unsqueeze(1) -> (N, 1)
        # gather selecciona el log_prob correspondiente a la clase correcta
        ce_loss = F.nll_loss(log_probs, targets, reduction='none')

        # Probabilidad de la clase correcta
        probs = torch.exp(log_probs)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        # Focal weight: (1 - pt)^gamma
        focal_weight = (1 - pt) ** self.gamma

        # Aplicar focal weight
        focal_loss = focal_weight * ce_loss

        # Aplicar alpha (peso por clase) si está definido
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            focal_loss = alpha_t * focal_loss

        # Reducción
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
import torch
import torch.nn as nn

class AsymmetricLoss(nn.Module):
    """
    Asymmetric Loss for imbalanced classification.
    
    Treats positives and negatives differently with asymmetric focusing.
    Better than Focal Loss for datasets with moderate imbalance.
    
    Parameters:
    -----------
    gamma_neg : float
        Focusing parameter for negative class (non-defective)
        Higher = focus more on hard negatives
    gamma_pos : float
        Focusing parameter for positive class (defective)
        Usually lower than gamma_neg
    clip : float
        Clipping value to prevent overconfidence on negatives
        Helps reduce false negatives
    reduction : str
        'mean' or 'sum'
    """
    def __init__(self, gamma_neg=2, gamma_pos=1, clip=0.05, reduction='mean'):
        super(AsymmetricLoss, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.reduction = reduction

    def forward(self, logits, targets):
        """
        logits: raw model outputs, shape (N, 2) for binary classification
        targets: ground truth labels, shape (N,) with values 0 or 1
        """
        # Get probabilities from logits
        probs = torch.softmax(logits, dim=1)
        
        # Probabilities for positive class (defective)
        pos_probs = probs[:, 1]  # P(class=1)
        neg_probs = probs[:, 0]  # P(class=0)
        
        # Convert targets to float for calculations
        targets_float = targets.float()
        
        # Asymmetric clipping on negative class
        # This prevents the model from being overconfident on negatives
        neg_probs = (neg_probs + self.clip).clamp(max=1.0)
        
        # Calculate cross-entropy components
        pos_loss = targets_float * torch.log(pos_probs.clamp(min=1e-8))
        neg_loss = (1 - targets_float) * torch.log(neg_probs.clamp(min=1e-8))
        
        # Asymmetric focusing: weight by (1-p)^gamma
        # Focus more on hard examples (low probability for correct class)
        pos_weight = ((1 - pos_probs) ** self.gamma_pos) * targets_float
        neg_weight = ((1 - neg_probs) ** self.gamma_neg) * (1 - targets_float)
        
        # Combine losses with asymmetric weighting
        loss = -(pos_loss * pos_weight + neg_loss * neg_weight)
        
        # Reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

## Freeze functions and optimizer builder

In [ ]:
def apply_freezing(model, layer_groups: list[str], layers_to_modify: list[str], freeze_flag: bool):
    """
    freeze_flag: True → freeze layers, False → unfreeze layers
    layers_to_modify: list of keys from layer_groups,
    """

    target_substrings = []
    for key in layers_to_modify:
        target_substrings += layer_groups[key]

    for name, module in model.named_modules():
        if any(substr in name for substr in target_substrings):
            for param in module.parameters():
                param.requires_grad = (not freeze_flag)  # True→freeze means requires_grad=False

def build_optimizer(model, lr: float, wd: float,
                    optimizer_type: str = "AdamW"):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    match optimizer_type:
        case "AdamW":
            return torch.optim.AdamW(
                trainable_params,
                lr=lr,
                weight_decay=wd
            )
        case "Adam":
            return torch.optim.Adam(
                trainable_params,
                lr=lr,
                weight_decay=wd
            )
        case _:
            raise "Wrong optimizer type"
        
# Optimizer and Criterion
def compute_class_stats_from_loader(loader):
    total_per_class = torch.zeros(2)

    for _, labels in loader:
        for c in [0, 1]:
            total_per_class[c] += (labels == c).sum()

    N0 = int(total_per_class[0].item())  # non-defective
    N1 = int(total_per_class[1].item())  # defective

    return N0, N1

### Train the model with cross validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
import numpy as np

# Number of channels [R, G, GRAY, W]
print(model_channels)
num_channels = len(model_channels)

# Early stopping parameters
patience = 30
min_delta = 1.5e-3
dropout_rate = 0.5
optimizer_type = "AdamW"
num_epochs = 120

best_val_aucpr = 0.0
epochs_no_improve = 0
best_model_state = None

# Params for Stratified K-Fold
k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, 
                              shuffle=True)
labels = np.array(dataset.labels)

# Parameters for the criterion 
weight_decay = 2e-3
alpha = torch.tensor([0.35, 0.65]).to(device)  # suma = 1.0

# Parameters for dataset
batch_size = 8

# Layers of Resnet34 and RESNET18
layer_groups = {
    "conv1": ["conv1", "bn1"],
    "layer1": ["layer1"],
    "layer2": ["layer2"],
    "layer3": ["layer3"],
    "layer4": ["layer4"],
    "head":   ["fc"],   # classifier
}

# Layers of EfficientNet-B0
# layer_groups = {
#     "stem": ["features.0"],
#     "early": ["features.1", "features.2"],      # Early stages
#     "mid": ["features.3", "features.4"],        # Middle stages
#     "late": ["features.5", "features.6"],       # Late stages
#     "final": ["features.7", "features.8"],      # Final blocks
#     "head": ["classifier"],
# }

model_head = nn.Sequential(
    # First layer: 512 -> 256
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.55),  # Increased from your 0.35 to combat overfitting
    
    # Second layer: 256 -> 128
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.40),  # Progressive dropout reduction
    
    # Output layer: 128 -> 2
    nn.Linear(128, 2)
)

# Dynamically freeze layers
# ResNet34 and RESNET18
freeze_schedule = [
    (20, True, ['layer1', 'layer2', 'layer3', 'layer4'], 8e-5),   # Only train head and conv1
    (100, False, ['layer1', 'layer2', 'layer3', 'layer4'], 3e-5),  # Train all
]

# EfficientNet-B0
# freeze_schedule = [
#     # Phase 1: Warmup - Train only head and stem (40 epochs)
#     # This lets the model adapt to your domain while keeping pretrained features intact
#     (40, True, ['early', 'mid', 'late', 'final'], 1e-4),
    
#     # Phase 2: Unfreeze final stages (50 epochs)
#     # Start fine-tuning the deepest semantic features
#     (50, False, ['final'], 2e-5),
    
#     # Phase 3: Unfreeze late stages (50 epochs)
#     # Add more capacity for fine-grained features
#     (50, False, ['late', 'final'], 1e-5),
    
#     # Phase 4: Unfreeze middle stages (60 epochs)
#     # Full feature hierarchy tuning
#     (60, False, ['mid', 'late', 'final'], 5e-6),
    
#     # Phase 5: Fine-tune everything (80 epochs)
#     # Polish all layers with very low learning rate
#     (80, False, ['early', 'mid', 'late', 'final'], 2e-6),
# ]

In [ ]:
from torch.utils.data import DataLoader

fold_results_aucpr = []
fold_results_f1 = []
fold_models = []  # Stores the best model for each fold

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n=== Fold {fold+1}/{k_folds} ===")

    # Subsets
    train_subset = torch.utils.data.Subset(dataset, train_idx)
    val_subset   = torch.utils.data.Subset(dataset, val_idx)

    # Apply transforms
    train_subset.dataset.transform = data_transform
    val_subset.dataset.transform   = data_transform

    # DataLoaders
    train_loader = DataLoader(train_subset, 
                              batch_size=batch_size, 
                              pin_memory=True, 
                              shuffle=True,
                              drop_last=False,
                              num_workers= 0
                              )
    val_loader   = DataLoader(val_subset, 
                              batch_size=batch_size, 
                              pin_memory=True, 
                              shuffle=False,
                              num_workers= 0
                              )

    # Fresh Model 
    k_model = build_model_resnet18(head=model_head, 
                                   n_channels=num_channels,            # 4
                                   selected_channels=model_channels,   # [R, G, GRAY, W]
                                   dropout_rate=dropout_rate).to(device)
    
    # Criterion
    # CrossEntropyLoss
    N0, N1 = compute_class_stats_from_loader(train_loader) # Get weights from the actual train loader

    # Calculate the weights
    beta = 0.999
    w0 = (1 - beta) / (1 - beta**N0)
    w1 = (1 - beta) / (1 - beta**N1)

    weights = torch.tensor([w0, w1]).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.66]).to(device), label_smoothing=0.07)

    # FocalLoss
    # criterion = FocalLoss(
    #     alpha=alpha,
    #     gamma=2.0,
    #     label_smoothing=0.0
    # )

    # Early stopping variables
    best_val_aucpr = 0.0
    best_val_f1 = 0.0
    best_train_aucpr = 0.0
    best_train_f1 = 0.0
    best_epoch = 0
    epochs_no_improve = 0
    global_epoch = 0

    # repeat_num = fold // 5 + 1
    # fold_num = fold % 5 + 1

    # print(f"\n=== Repeat {repeat_num}/3 | Fold {fold_num}/5 ===")

    for phase_id, (phase_epochs, freeze_flag, layers_to_modify, lr) in enumerate(freeze_schedule):
        print(
            f"\n  Phase {phase_id+1}: "
            f"{'Freeze' if freeze_flag else 'Unfreeze'} {layers_to_modify} | "
            f"epochs={phase_epochs} | lr={lr}"
        )

        # Apply freezing
        apply_freezing(k_model, layer_groups, layers_to_modify, freeze_flag)

        # Restart optimizer
        optimizer = build_optimizer(k_model, lr, weight_decay, optimizer_type=optimizer_type)

        # Cosine scheduler (per phase)
        # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        #     optimizer, T_max=phase_epochs
        # )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=30,
            T_mult=2,
            eta_min=1e-7
        )

        # Training loop (phase)
        for epoch in range(phase_epochs):
            global_epoch += 1

            train_loss, train_aucpr, train_f1 = Train(
                k_model, train_loader, optimizer, criterion
            )

            val_loss, val_aucpr, val_f1, val_best_th = Validate(
                k_model, val_loader, criterion
            )

            scheduler.step()

            # ---- EARLY STOPPING ----
            if val_aucpr > best_val_aucpr + min_delta:
                best_val_aucpr = val_aucpr
                best_val_f1 = val_f1
                best_train_aucpr = train_aucpr
                best_train_f1 = train_f1
                best_epoch = global_epoch
                epochs_no_improve = 0
                best_model_state = k_model.state_dict()
            else:
                epochs_no_improve += 1

            # Light progress print
            print(
                f"    Epoch {global_epoch:03d} | "
                f"Train AUC-PR={train_aucpr:.4f} | "
                f"Val AUC-PR={val_aucpr:.4f} | "
                f"NoImprove={epochs_no_improve}/{patience}",
                end="\r" # Remove last line before printing the next one
            )

            if epochs_no_improve >= patience:
                print("\n    Early stopping triggered.")
                break

        if epochs_no_improve >= patience:
            break

    # Load best model for this fold
    k_model.load_state_dict(best_model_state)
    k_model.eval()

    # Fold summary
    fold_results_aucpr.append(best_val_aucpr)
    fold_results_f1.append(best_val_f1)
    fold_models.append(k_model.cpu())

    print(
        f"\nFold {fold+1} DONE | "
        f"Epochs={best_epoch} | "
        f"Train AUC-PR={best_train_aucpr:.4f} | "
        f"Val AUC-PR={best_val_aucpr:.4f} | "
        f"Val F1={best_val_f1:.4f}"
    )

In [ ]:
fold_results_aucpr = np.array(fold_results_aucpr)
fold_results_f1 = np.array(fold_results_f1)

print(f"\nCV AUC-PR: {fold_results_aucpr.mean():.4f} ± {fold_results_aucpr.std():.4f}")
print(f"\nCV F1 score: {fold_results_f1.mean():.4f} ± {fold_results_f1.std():.4f}")

### Export models in ensemble, results from CV. 
#### Add TTA for improving the performance

In [ ]:
class EnsembleTTAModel(nn.Module):
    """
    Flexible ensemble with configurable TTA transformations
    Allows dynamic selection of which augmentations to apply
    """
    def __init__(self, models, selected_channels=None, tta_config=None):
        """
        Args:
            models: List of trained fold models
            selected_channels: List of channel indices to use
            tta_config: Dict or list specifying TTA transforms
                        If dict: {'flips': True, 'rotations': True, 'combined': False}
                        If list: ['original', 'h_flip', 'v_flip', 'rot_90', ...]
                        If None or int: Use default with that many transforms
        """
        super().__init__()
        self.models = nn.ModuleList(models)
        
        # Channel selection
        if selected_channels is not None:
            self.selected_channels = list(selected_channels)
        else:
            self.selected_channels = None
        
        # Parse TTA configuration
        self.tta_config = self._parse_tta_config(tta_config)
        self.n_tta = len(self.tta_config)
    
    def _parse_tta_config(self, tta_config):
        """
        Parse TTA configuration into list of transform names
        
        Returns: List of transform names to apply
        """
        # Default configurations
        PRESETS = {
            'none': [],
            'minimal': ['original', 'h_flip', 'v_flip'],
            'flips': ['original', 'h_flip', 'v_flip', 'hv_flip'],
            'standard': ['original', 'h_flip', 'v_flip', 'hv_flip', 'rot_90', 'rot_270'],
            'full': ['original', 'h_flip', 'v_flip', 'hv_flip', 'rot_90', 'rot_180', 'rot_270'],
            'maximum': ['original', 'h_flip', 'v_flip', 'hv_flip', 'rot_90', 'rot_180', 'rot_270', 'rot_90_h_flip'],
        }
        
        # Case 1: String preset
        if isinstance(tta_config, str):
            if tta_config in PRESETS:
                return PRESETS[tta_config]
            else:
                raise ValueError(f"Unknown preset: {tta_config}. Options: {list(PRESETS.keys())}")
        
        # Case 2: Integer (number of transforms)
        elif isinstance(tta_config, int):
            all_transforms = ['original', 'h_flip', 'v_flip', 'hv_flip', 'rot_90', 'rot_180', 'rot_270', 'rot_90_h_flip']
            return all_transforms[:tta_config]
        
        # Case 3: List of transform names
        elif isinstance(tta_config, list):
            return tta_config
        
        # Case 4: Dict configuration
        elif isinstance(tta_config, dict):
            transforms = ['original']  # Always include original
            
            if tta_config.get('h_flip', True):
                transforms.append('h_flip')
            if tta_config.get('v_flip', True):
                transforms.append('v_flip')
            if tta_config.get('hv_flip', True):
                transforms.append('hv_flip')
            if tta_config.get('rotations', False):
                transforms.extend(['rot_90', 'rot_180', 'rot_270'])
            if tta_config.get('combined', False):
                transforms.append('rot_90_h_flip')
            
            return transforms
        
        # Case 5: None (default)
        else:
            return PRESETS['standard']  # Default to standard
    
    def rotate_90(self, x):
        """90° clockwise rotation"""
        return torch.flip(x.transpose(2, 3), dims=[2])
    
    def rotate_180(self, x):
        """180° rotation"""
        return torch.flip(x, dims=[2, 3])
    
    def rotate_270(self, x):
        """270° clockwise rotation"""
        return torch.flip(x.transpose(2, 3), dims=[3])
    
    def apply_transform(self, x, transform_name):
        """
        Apply single transformation by name
        
        Args:
            x: Input tensor
            transform_name: Name of transformation to apply
        
        Returns:
            Transformed tensor
        """
        transforms = {
            'original': lambda t: t,
            'h_flip': lambda t: torch.flip(t, dims=[3]),
            'v_flip': lambda t: torch.flip(t, dims=[2]),
            'hv_flip': lambda t: torch.flip(t, dims=[2, 3]),
            'rot_90': self.rotate_90,
            'rot_180': self.rotate_180,
            'rot_270': self.rotate_270,
            'rot_90_h_flip': lambda t: torch.flip(self.rotate_90(t), dims=[3]),
            'rot_90_v_flip': lambda t: torch.flip(self.rotate_90(t), dims=[2]),
            'rot_270_h_flip': lambda t: torch.flip(self.rotate_270(t), dims=[3]),
        }
        
        if transform_name not in transforms:
            raise ValueError(f"Unknown transform: {transform_name}. Options: {list(transforms.keys())}")
        
        return transforms[transform_name](x)
    
    def tta_transforms(self, x):
        """
        Generate all configured TTA transformations
        
        Returns:
            List of transformed tensors
        """
        return [self.apply_transform(x, name) for name in self.tta_config]
    
    def forward(self, x):
        """
        Forward pass with ensemble and TTA
        
        Args:
            x: Input tensor [B, 6, H, W]
        
        Returns:
            Log probabilities [B, 2]
        """
        # Select channels
        if self.selected_channels is not None:
            x = x[:, self.selected_channels, :, :]
        
        # Generate augmented versions
        augmented_inputs = self.tta_transforms(x)
        
        # Run all models on all augmentations
        all_predictions = []
        for aug_input in augmented_inputs:
            for model in self.models:
                model.eval()
                logits = model(aug_input)
                probs = F.softmax(logits, dim=1)
                all_predictions.append(probs)
        
        # Average all predictions
        avg_probs = torch.stack(all_predictions).mean(dim=0)
        return torch.log(avg_probs + 1e-8)
    
    def get_tta_info(self):
        """Get information about current TTA configuration"""
        return {
            'n_models': len(self.models),
            'n_tta': self.n_tta,
            'total_predictions': len(self.models) * self.n_tta,
            'transforms': self.tta_config,
        }

In [ ]:
print("\n" + "="*60)
print("BUILDING ENSEMBLE MODEL")
print("="*60)

# Move models back to device and eval mode
for i, model in enumerate(fold_models):
    fold_models[i] = model.to(device)
    fold_models[i].eval()

# Create ensemble with TTA
ensemble_model = EnsembleTTAModel(
    models=fold_models,
    selected_channels=model_channels,  # Your selected channels
    # tta_config='full'  # Type of augmentations in test
    tta_config='flips'  # Type of augmentations in test
).to(device)

ensemble_model.eval()

print(f"✅ Ensemble created:")
print(f"   - {len(fold_models)} fold models")
print(f"   - {5} TTA transforms")
print(f"   - Total predictions averaged: {len(fold_models) * 5}")
print(f"   - Selected channels: {model_channels}")

In [ ]:
print("\n" + "="*60)
print("EXPORTING TO ONNX")
print("="*60)

# Determine input size based on whether we're using channel selection
if model_channels is not None:
    # ONNX will accept 6 channels, select internally
    dummy_input_channels = 6
else:
    dummy_input_channels = num_channels

# Create dummy input
dummy_input = torch.randn(1, dummy_input_channels, 310, 310).to(device)
# Test forward pass
with torch.no_grad():
    test_output = ensemble_model(dummy_input)
    test_prob = F.softmax(test_output, dim=1)[0, 1].item()
    print(f"✅ Test forward pass successful")
    print(f"   Dummy prediction: {test_prob:.4f}")

# Export to ONNX
onnx_filename = "ensemble_model.onnx"

torch.onnx.export(
    ensemble_model,
    dummy_input,
    onnx_filename,
    # export_params=True,
    opset_version=14,
    input_names=['input'],
    output_names=['output'],
    do_constant_folding=True,
    # dynamic_axes={
        # 'input': {0: 'batch_size'},
        # 'output': {0: 'batch_size'}
    # },
    # verbose=False
)

print(f"\n✅ ONNX model exported: {onnx_filename}")
print(f"\nModel specifications:")
print(f"   - Input:  [batch, {dummy_input_channels}, 310, 310]")
print(f"   - Output: [batch, 2] (logits)")
print(f"   - Contains: {len(fold_models)} models × {5} TTA = {len(fold_models) * 5} predictions")

# Get file size
import os
file_size_mb = os.path.getsize(onnx_filename) / (1024 * 1024)
print(f"   - File size: {file_size_mb:.1f} MB")


# Verify ONNX Model

print("\n" + "="*60)
print("VERIFYING ONNX MODEL")
print("="*60)

try:
    import onnxruntime as ort
    
    # Load ONNX model
    ort_session = ort.InferenceSession(onnx_filename)
    
    # Test with same dummy input
    ort_inputs = {ort_session.get_inputs()[0].name: dummy_input.cpu().numpy()}
    ort_outputs = ort_session.run(None, ort_inputs)
    ort_logits = ort_outputs[0]
    
    # Compare with PyTorch output
    pytorch_logits = test_output.cpu().numpy()
    
    max_diff = np.abs(pytorch_logits - ort_logits).max()
    
    if max_diff < 1e-4:
        print(f"✅ ONNX verification PASSED")
        print(f"   Max difference: {max_diff:.2e}")
    else:
        print(f"⚠️  ONNX verification: Large difference detected")
        print(f"   Max difference: {max_diff:.2e}")
    
    # Show prediction
    ort_probs = np.exp(ort_logits) / np.sum(np.exp(ort_logits), axis=1, keepdims=True)
    print(f"   ONNX prediction: {ort_probs[0, 1]:.4f}")
    
except ImportError:
    print("⚠️  onnxruntime not installed. Install with: pip install onnxruntime")
except Exception as e:
    print(f"❌ ONNX verification failed: {e}")

### Testing the model

In [ ]:
# ============================================
# VERIFICAR QUE ONNX == PYTORCH
# ============================================
import onnxruntime as ort
import numpy as np
import torch
import torch.nn.functional as F

# Cargar ONNX
ort_session = ort.InferenceSession("ensemble_model.onnx", providers=["CPUExecutionProvider"])

# Comparar predicciones
print("Comparando PyTorch vs ONNX:")
ensemble_model.eval()
for i in range(5):
    test_input = torch.rand(1, 6, 310, 310) # Original size
    
    # PyTorch
    with torch.no_grad():
        pt_logits = ensemble_model(test_input.to(device))
        # pt_logits = wrapped_model(test_input.cpu())
        pt_prob = F.softmax(pt_logits, dim=1)[0, 1].item()
    
    # ONNX
    onnx_out = ort_session.run(None, {"input": test_input.numpy()})[0]
    onnx_prob = (np.exp(onnx_out) / np.exp(onnx_out).sum())[0, 1]
    
    print(f"   Test {i+1}: PyTorch={pt_prob:.4f}, ONNX={onnx_prob:.4f}, Diff={abs(pt_prob-onnx_prob):.6f}")

# Si Diff > 0.01 hay problema